## Daily Project Recap

### Yesterday — Data Preparation

We explored and cleaned the customer-review dataset.

Main decisions:
- Removed rows missing review text or rating.
- Normalized review-text whitespace.
- Removed duplicate review texts after checking that duplicate ratings were consistent.
- Created three sentiment labels from ratings:
  - 1–2 stars → Negative
  - 3 stars → Neutral
  - 4–5 stars → Positive
- Final cleaned dataset: 34,615 reviews.
- We identified a strong class imbalance toward Positive reviews.

### Today — Sentiment Classification

Our goal is to compare several sentiment classifiers fairly using the same cleaned dataset, stratified train/test split, and TF-IDF features.

Models being evaluated:
- Logistic Regression
- Multinomial Naive Bayes
- Linear SVM
- Balanced Linear SVM

We will compare accuracy, precision, recall, F1 and Macro F1.

Because the dataset is highly imbalanced, we will not select the final model based only on accuracy.

### Team Coordination

Caio:
- Rebuilding the reproducible sentiment-analysis notebook.
- Logistic Regression and Naive Bayes experiments.
- Model comparison and documentation.

Diogo:
- Linear SVM experiments.
- Class-imbalance and feature-representation experiments.
- Beginning clustering exploration.

Shared decisions:
- Use the same train/test split and TF-IDF configuration for fair model comparison.
- Select one final sentiment classifier based on balanced performance.
- For clustering, use `reviews.text` as the model input and keep metadata only for later cluster interpretation.

# Day 2 — Sentiment Analysis

## Goal

Build and evaluate a three-class sentiment classifier for customer reviews.

The target classes are:

- Negative
- Neutral
- Positive

Because the dataset is highly imbalanced toward Positive reviews, accuracy alone is not enough to evaluate model quality.

We will compare several classifiers using the same cleaned dataset, train/test split, and TF-IDF representation.

## Definition of Done

- Load and validate the cleaned Day 1 dataset.
- Create a reproducible 80/20 stratified train/test split.
- Convert review text into TF-IDF features.
- Establish a majority-class baseline.
- Train Logistic Regression.
- Train Multinomial Naive Bayes.
- Train Linear SVM.
- Train Balanced Linear SVM.
- Compare accuracy, precision, recall, F1, and macro F1.
- Inspect confusion matrices and minority-class errors.
- Select one final sentiment classifier based on the evidence.

In [3]:
# Import the tools needed for Day 2.

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report
)

from sklearn.model_selection import train_test_split

## 1. Load and validate the cleaned dataset

### Question

Are we starting the sentiment analysis with the correct cleaned dataset from Day 1?

### What we did

We load the cleaned review dataset and verify that it has the expected number of rows and the columns required for sentiment modeling.

### Why this matters

If the wrong or an outdated dataset is loaded, all later model results would be unreliable.

In [4]:
# Find the current folder where the notebook is running.
PROJECT_ROOT = Path.cwd().resolve()

# If the current folder does not contain "datasets",
# move one folder up to the project root.
if not (PROJECT_ROOT / "datasets").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

# Build the path to the cleaned dataset created on Day 1.
CLEAN_DATA_PATH = PROJECT_ROOT / "datasets" / "clean_reviews.csv"

# Store the expected number of cleaned rows.
EXPECTED_ROWS = 34615

# Stop the notebook if the cleaned dataset cannot be found.
assert CLEAN_DATA_PATH.exists(), (
    f"Cleaned dataset not found at {CLEAN_DATA_PATH}. "
    "Run the Day 1 cleaning notebook first."
)

# Load the cleaned dataset into a Pandas DataFrame.
df = pd.read_csv(
    CLEAN_DATA_PATH,
    low_memory=False
)

# Display the number of rows and columns.
df.shape

(34615, 7)

In [5]:
# Define the columns required for sentiment analysis.
required_columns = {
    "reviews.text",
    "reviews.rating",
    "sentiment"
}

# Find whether any required columns are missing.
missing_columns = required_columns.difference(df.columns)

# Stop if any required column is missing.
assert not missing_columns, (
    f"Missing required columns: {missing_columns}"
)

# Confirm that the number of rows matches the expected cleaned dataset.
assert len(df) == EXPECTED_ROWS, (
    f"Expected {EXPECTED_ROWS:,} rows, found {len(df):,}"
)

# Confirm there are no missing values in the critical columns.
assert df[
    ["reviews.text", "reviews.rating", "sentiment"]
].isna().sum().sum() == 0

print("Data validation passed.")

Data validation passed.


## 2. Define X and y

### Question

What information will the model use, and what are we asking it to predict?

### What we did

We define:

- `X` as the customer review text.
- `y` as the sentiment label.

The model will learn patterns in the review text that help predict whether a review is Negative, Neutral, or Positive.

In [6]:
# X contains the input text that the model will learn from.
X = df["reviews.text"]

# y contains the correct sentiment label for each review.
y = df["sentiment"]

# Check how many examples are in X and y.
print("Number of reviews in X:", len(X))
print("Number of labels in y:", len(y))

Number of reviews in X: 34615
Number of labels in y: 34615


In [ ]:
## 3. Create training and test sets

### Question

How can we evaluate whether the model works on reviews it has never seen before?

### What we did

We split the dataset into:

- 80% training data
- 20% test data

The model learns from the training set.

The test set stays unseen until evaluation so we can measure how well the model generalizes to new reviews.

In [7]:
# Split the review texts and sentiment labels into training and test sets.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Check how many reviews are in each part.
print("Training reviews:", len(X_train))
print("Test reviews:", len(X_test))

# Confirm that the split still contains every original review.
print("Total reviews:", len(X_train) + len(X_test))

Training reviews: 27692
Test reviews: 6923
Total reviews: 34615


In [ ]:
## 4. Convert review text into TF-IDF features

### Question

How can a machine-learning model work with text?

### What we did

We convert the review text into numerical features using TF-IDF.

TF-IDF gives more importance to words and phrases that are useful for distinguishing reviews, while reducing the importance of very common terms.

In [8]:
# Create the TF-IDF vectorizer.
# max_features=10000 limits the vocabulary to the 10,000 most useful features.
# ngram_range=(1, 2) includes both single words and two-word phrases.
tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2)
)

# Learn the vocabulary only from the training reviews
# and convert the training text into numerical features.
X_train_tfidf = tfidf.fit_transform(X_train)

# Convert the test reviews using the vocabulary
# already learned from the training data.
X_test_tfidf = tfidf.transform(X_test)

# Check the dimensions of the feature matrices.
print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Test TF-IDF shape:", X_test_tfidf.shape)

Training TF-IDF shape: (27692, 10000)
Test TF-IDF shape: (6923, 10000)


## 5. Create a majority-class baseline

### Question

How good would a model look if it learned nothing and simply predicted the most common sentiment every time?

### What we did

We train a `DummyClassifier` that always predicts the majority class.

This gives us a baseline to compare against our real models.

Because the dataset is strongly imbalanced toward Positive reviews, this baseline helps show why accuracy alone can be misleading.

In [10]:
# Create a dummy classifier that always predicts
# the most common class in the training data.
dummy_model = DummyClassifier(
    strategy="most_frequent"
)

# Train the dummy model.
# It does not learn text patterns;
# it only learns which class is most common.
dummy_model.fit(
    X_train_tfidf,
    y_train
)

# Make predictions on the unseen test set.
dummy_predictions = dummy_model.predict(
    X_test_tfidf
)

# Calculate the overall accuracy.
dummy_accuracy = accuracy_score(
    y_test,
    dummy_predictions
)

print("Dummy model accuracy:", round(dummy_accuracy, 4))

Dummy model accuracy: 0.9333


In [11]:
# Show precision, recall, and F1-score
# for each sentiment class.
print(
    classification_report(
        y_test,
        dummy_predictions,
        zero_division=0
    )
)

              precision    recall  f1-score   support

    negative       0.00      0.00      0.00       162
     neutral       0.00      0.00      0.00       300
    positive       0.93      1.00      0.97      6461

    accuracy                           0.93      6923
   macro avg       0.31      0.33      0.32      6923
weighted avg       0.87      0.93      0.90      6923



In [ ]:
## 6. Logistic Regression baseline

### Question

Can a real classifier learn useful sentiment patterns from the TF-IDF features?

### What we did

We train Logistic Regression using the same training data and TF-IDF representation used in the baseline comparison.

Then we evaluate it on the unseen test set using accuracy, precision, recall, F1, and macro F1.

In [12]:
# Create the Logistic Regression classifier.
# max_iter=1000 gives the model enough training iterations to converge.
logistic_model = LogisticRegression(
    max_iter=1000
)

# Train the model using the TF-IDF training features
# and the correct sentiment labels.
logistic_model.fit(
    X_train_tfidf,
    y_train
)

# Predict sentiment for the unseen test reviews.
logistic_predictions = logistic_model.predict(
    X_test_tfidf
)

# Calculate the overall accuracy.
logistic_accuracy = accuracy_score(
    y_test,
    logistic_predictions
)

print(
    "Logistic Regression accuracy:",
    round(logistic_accuracy, 4)
)

Logistic Regression accuracy: 0.9363


In [13]:
# Show precision, recall, and F1-score
# for each sentiment class.
print(
    classification_report(
        y_test,
        logistic_predictions,
        zero_division=0
    )
)

              precision    recall  f1-score   support

    negative       0.62      0.15      0.25       162
     neutral       0.50      0.05      0.09       300
    positive       0.94      1.00      0.97      6461

    accuracy                           0.94      6923
   macro avg       0.69      0.40      0.43      6923
weighted avg       0.91      0.94      0.91      6923



### Result

Logistic Regression achieved 93.63% accuracy, only slightly higher than the 93.33% majority-class baseline.

However, its Macro F1 increased from 0.32 to 0.43 because the model began identifying some Negative and Neutral reviews.

### Decision / Finding

Accuracy alone remains misleading because Positive reviews dominate the dataset.

Logistic Regression learns useful sentiment patterns, but performance on the minority Negative and Neutral classes is still weak.

In [ ]:
## 7. Multinomial Naive Bayes

### Question

Can a probabilistic text classifier perform better than Logistic Regression on this sentiment task?

### What we did

We train Multinomial Naive Bayes using the same TF-IDF training and test data.

Using the same data and features keeps the comparison fair.

In [14]:
# Create the Multinomial Naive Bayes classifier.
naive_bayes_model = MultinomialNB()

# Train the model using the TF-IDF training features
# and the correct sentiment labels.
naive_bayes_model.fit(
    X_train_tfidf,
    y_train
)

# Predict sentiment for the unseen test reviews.
naive_bayes_predictions = naive_bayes_model.predict(
    X_test_tfidf
)

# Calculate the overall accuracy.
naive_bayes_accuracy = accuracy_score(
    y_test,
    naive_bayes_predictions
)

print(
    "Naive Bayes accuracy:",
    round(naive_bayes_accuracy, 4)
)

Naive Bayes accuracy: 0.9333


In [15]:
# Show precision, recall, and F1-score
# for each sentiment class.
print(
    classification_report(
        y_test,
        naive_bayes_predictions,
        zero_division=0
    )
)

              precision    recall  f1-score   support

    negative       0.00      0.00      0.00       162
     neutral       0.00      0.00      0.00       300
    positive       0.93      1.00      0.97      6461

    accuracy                           0.93      6923
   macro avg       0.31      0.33      0.32      6923
weighted avg       0.87      0.93      0.90      6923



### Result

Multinomial Naive Bayes achieved 93.33% accuracy, but this was misleading.

The model failed to identify any Negative or Neutral reviews:

- Negative recall: 0.00
- Neutral recall: 0.00
- Macro F1: 0.32

### Decision / Finding

Naive Bayes is not suitable as the final classifier in its current configuration.

Despite high accuracy, its behavior is almost identical to the majority-class baseline.

## 8. Linear SVM

### Question

Can Linear SVM improve minority-class performance while maintaining strong overall accuracy?

### What we did

We train Linear SVM using the same TF-IDF features, training data, and test data used by the previous models.

In [16]:
# Create the Linear SVM classifier.
linear_svm_model = LinearSVC(
    random_state=42
)

# Train the model using the TF-IDF training features.
linear_svm_model.fit(
    X_train_tfidf,
    y_train
)

# Predict sentiment for the unseen test reviews.
linear_svm_predictions = linear_svm_model.predict(
    X_test_tfidf
)

# Calculate the overall accuracy.
linear_svm_accuracy = accuracy_score(
    y_test,
    linear_svm_predictions
)

print(
    "Linear SVM accuracy:",
    round(linear_svm_accuracy, 4)
)

Linear SVM accuracy: 0.9359


In [17]:
# Show precision, recall, and F1-score
# for each sentiment class.
print(
    classification_report(
        y_test,
        linear_svm_predictions,
        zero_division=0
    )
)

              precision    recall  f1-score   support

    negative       0.51      0.25      0.33       162
     neutral       0.38      0.11      0.17       300
    positive       0.95      0.99      0.97      6461

    accuracy                           0.94      6923
   macro avg       0.61      0.45      0.49      6923
weighted avg       0.91      0.94      0.92      6923



### Result

Linear SVM achieved approximately 93.59% accuracy.

Its overall accuracy was very similar to Logistic Regression, but performance on the minority classes improved:

- Negative recall increased from 0.15 to 0.25.
- Neutral recall increased from 0.05 to 0.11.
- Macro F1 increased from 0.43 to 0.49.

### Decision / Finding

Linear SVM is a stronger classifier than Logistic Regression for this imbalanced dataset because it improves Negative and Neutral detection while maintaining similar overall accuracy.

## 9. Balanced Linear SVM

### Question

Can we improve the detection of rare Negative and Neutral reviews by giving those classes more importance during training?

### What we did

We retrain Linear SVM using `class_weight="balanced"`.

This tells the model to give more importance to underrepresented classes during training.

In [18]:
# Create a balanced Linear SVM classifier.
# class_weight="balanced" increases the importance
# of the rare Negative and Neutral classes.
balanced_svm_model = LinearSVC(
    class_weight="balanced",
    random_state=42
)

# Train the balanced model using the same TF-IDF features.
balanced_svm_model.fit(
    X_train_tfidf,
    y_train
)

# Predict sentiment for the unseen test reviews.
balanced_svm_predictions = balanced_svm_model.predict(
    X_test_tfidf
)

# Calculate the overall accuracy.
balanced_svm_accuracy = accuracy_score(
    y_test,
    balanced_svm_predictions
)

print(
    "Balanced Linear SVM accuracy:",
    round(balanced_svm_accuracy, 4)
)

Balanced Linear SVM accuracy: 0.9234


In [19]:
# Create a balanced Linear SVM classifier.
# class_weight="balanced" increases the importance
# of the rare Negative and Neutral classes.
balanced_svm_model = LinearSVC(
    class_weight="balanced",
    random_state=42
)

# Train the balanced model using the same TF-IDF features.
balanced_svm_model.fit(
    X_train_tfidf,
    y_train
)

# Predict sentiment for the unseen test reviews.
balanced_svm_predictions = balanced_svm_model.predict(
    X_test_tfidf
)

# Calculate the overall accuracy.
balanced_svm_accuracy = accuracy_score(
    y_test,
    balanced_svm_predictions
)

print(
    "Balanced Linear SVM accuracy:",
    round(balanced_svm_accuracy, 4)
)

Balanced Linear SVM accuracy: 0.9234


In [20]:
# Create a balanced Linear SVM.
# class_weight="balanced" gives more importance
# to classes that have fewer training examples.
balanced_svm_model = LinearSVC(
    class_weight="balanced",
    random_state=42
)

# Train the model using the same TF-IDF features.
balanced_svm_model.fit(
    X_train_tfidf,
    y_train
)

# Predict the sentiment of unseen test reviews.
balanced_svm_predictions = balanced_svm_model.predict(
    X_test_tfidf
)

# Calculate the overall accuracy.
balanced_svm_accuracy = accuracy_score(
    y_test,
    balanced_svm_predictions
)

print(
    "Balanced Linear SVM accuracy:",
    round(balanced_svm_accuracy, 4)
)

Balanced Linear SVM accuracy: 0.9234


In [21]:
# Display precision, recall, and F1-score
# for every sentiment class.
print(
    classification_report(
        y_test,
        balanced_svm_predictions,
        zero_division=0
    )
)

              precision    recall  f1-score   support

    negative       0.47      0.39      0.43       162
     neutral       0.29      0.28      0.28       300
    positive       0.96      0.97      0.96      6461

    accuracy                           0.92      6923
   macro avg       0.57      0.55      0.56      6923
weighted avg       0.92      0.92      0.92      6923



In [ ]:
## 10. Final Model Comparison

### Question

Which classifier gives the best overall performance for our highly imbalanced sentiment dataset?

### What we did

We compare all tested classifiers using the same test set.

Because Positive reviews dominate the dataset, we focus especially on Macro F1 and minority-class recall rather than accuracy alone.

In [22]:
# Store each model name together with its predictions.
# This lets us evaluate all models in the same way.
model_predictions = {
    "Dummy Baseline": dummy_predictions,
    "Logistic Regression": logistic_predictions,
    "Naive Bayes": naive_bayes_predictions,
    "Linear SVM": linear_svm_predictions,
    "Balanced Linear SVM": balanced_svm_predictions
}

# Create an empty list where we will store
# one summary row for each model.
comparison_rows = []

# Go through each model and its predictions.
for model_name, predictions in model_predictions.items():

    # Create a classification report as a Python dictionary
    # so we can extract individual metrics.
    report = classification_report(
        y_test,
        predictions,
        output_dict=True,
        zero_division=0
    )

    # Store the most useful metrics for our comparison.
    comparison_rows.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Macro F1": report["macro avg"]["f1-score"],
        "Negative Recall": report["negative"]["recall"],
        "Neutral Recall": report["neutral"]["recall"],
        "Positive Recall": report["positive"]["recall"]
    })

# Convert the list of results into a DataFrame.
comparison_df = pd.DataFrame(comparison_rows)

# Use the model name as the table index.
comparison_df = comparison_df.set_index("Model")

# Display the results rounded to three decimal places.
display(comparison_df.round(3))

,Accuracy,Macro F1,Negative Recall,Neutral Recall,Positive Recall
Model,,,,,
Dummy Baseline,0.933,0.322,0.000,0.000,1.000
Logistic Regression,0.936,0.434,0.154,0.047,0.997
Naive Bayes,0.933,0.322,0.000,0.000,1.000
Linear SVM,0.936,0.490,0.247,0.107,0.992
Balanced Linear SVM,0.923,0.558,0.389,0.280,0.967


### Final Decision

Balanced Linear SVM + TF-IDF is selected as the final sentiment classifier.

Although Logistic Regression achieved slightly higher accuracy, Balanced Linear SVM achieved the highest Macro F1 and substantially better recall for the underrepresented Negative and Neutral classes.

Because the dataset is highly imbalanced, this more balanced performance is more useful than maximizing accuracy alone.

**Selected sentiment model: Balanced Linear SVM + TF-IDF**

### Final Cluster Themes

The team aligned the cluster interpretations by semantic meaning rather than numerical cluster ID.

Final themes:

- Ease of Use & Setup
- General & Mixed Feedback
- Fire TV & Streaming
- Kids, Family & Gifts
- Kindle & Reading
- Tablets & Value

Cluster IDs may differ between KMeans runs, but the semantic themes are the relevant project output.